# Unified CSBS Scoring REDCap Validation

This notebook validates **both** CSBS instruments across their respective REDCap sandboxes:
- **CSBS Caregiver**: PID 6205 (NANO - CSBS Caregiver / EDI Scoring Sandbox)
- **CSBS BS**: PID 6207 (NANO Lab Assessments & Double Data Entry Sandbox)

It checks:
1. Auto-calculated score formulas match approved logic (no mismatches)
2. All raw/composite/total score fields are properly labeled with AUTO-CALCULATED prefix
3. Manual norm fields are present by visit (e.g., 12m/24m rule checks)
4. Field visibility and branching logic where applicable
5. Consistency across both instruments

In [1]:
import os
import requests
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP

API_URL = os.getenv('REDCAP_API_URL', 'https://redcap.research.sc.edu/api/')
TOKEN_6205 = os.getenv('REDCAP_TOKEN_6205', '').strip()  # CSBS Caregiver
TOKEN_6207 = os.getenv('REDCAP_TOKEN_6207', '7B81ED83E91654B3573A2D7FE4B223D2').strip()  # CSBS BS

if not TOKEN_6205:
    print('WARNING: REDCAP_TOKEN_6205 not set. CSBS Caregiver (PID 6205) validation will be skipped.')
    print('Set REDCAP_TOKEN_6205 to validate PID 6205.')
    TOKEN_6205 = None

if not TOKEN_6207:
    raise RuntimeError('REDCAP_TOKEN_6207 must be set for CSBS BS (PID 6207) validation.')

def post(token, content, **params):
    """Make API call to REDCap."""
    data = {'token': token, 'content': content, 'format': 'json', 'returnFormat': 'json'}
    data.update(params)
    r = requests.post(API_URL, data=data, timeout=180)
    r.raise_for_status()
    payload = r.json()
    if isinstance(payload, dict) and payload.get('error'):
        raise RuntimeError(payload['error'])
    return payload

print('\n=== STEP 1: Verify Projects ===')
if TOKEN_6207:
    proj_6207 = post(TOKEN_6207, 'project')
    proj_6207 = proj_6207[0] if isinstance(proj_6207, list) and proj_6207 else proj_6207
    print(f'✓ PID 6207: {proj_6207.get("project_title")}')
    if str(proj_6207.get('project_id')) != '6207':
        raise RuntimeError('Token does not map to PID 6207')

if TOKEN_6205:
    proj_6205 = post(TOKEN_6205, 'project')
    proj_6205 = proj_6205[0] if isinstance(proj_6205, list) and proj_6205 else proj_6205
    print(f'✓ PID 6205: {proj_6205.get("project_title")}')
    if str(proj_6205.get('project_id')) != '6205':
        raise RuntimeError('Token does not map to PID 6205')

Set REDCAP_TOKEN_6205 to validate PID 6205.

=== STEP 1: Verify Projects ===


✓ PID 6207: NANO Lab Assessments & Double Data Entry- Sandbox


In [2]:
# Utility functions
def num(v):
    if v in (None, ''):
        return None
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def half_up(v):
    return float(Decimal(str(v)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

def bs_expected(row):
    """Calculate expected CSBS BS scores from raw items."""
    required = [f'csbsbs_scale{i}' for i in range(1, 16)] + [
        'csbsbs_scale16_1', 'csbsbs_scale16_2', 'csbsbs_scale16_3',
        'csbsbs_scale17', 'csbsbs_scale18', 'csbsbs_scale19', 'csbsbs_scale20'
    ]
    if any(num(row.get(k)) is None for k in required):
        return None

    g = lambda name: num(row.get(name))
    emotion = half_up(g('csbsbs_scale1') + g('csbsbs_scale2') + 3 * g('csbsbs_scale3'))
    communication = half_up(g('csbsbs_scale4') / 3 + g('csbsbs_scale5') + g('csbsbs_scale6') + g('csbsbs_scale7'))
    gestures = half_up(2 * g('csbsbs_scale8') + g('csbsbs_scale9'))
    sounds = half_up(g('csbsbs_scale10') + 2 * g('csbsbs_scale11'))
    words = half_up(g('csbsbs_scale12') + g('csbsbs_scale13') / 2 + g('csbsbs_scale14') + g('csbsbs_scale15'))
    understanding = half_up(3 * (g('csbsbs_scale16_1') + g('csbsbs_scale16_2') + g('csbsbs_scale16_3')))
    object_use = half_up(g('csbsbs_scale17') + g('csbsbs_scale18') + g('csbsbs_scale19') + g('csbsbs_scale20'))

    return {
        'csbsbs_emotionraw': emotion,
        'csbsbs_comraw': communication,
        'csbsbs_gesraw': gestures,
        'csbsbs_soundsraw': sounds,
        'csbsbs_wordsraw': words,
        'csbsbs_underraw': understanding,
        'csbsbs_objectraw': object_use,
        'csbsbs_socialcompositecalc': emotion + communication + gestures,
        'csbsbs_speechcompositecalc': sounds + words,
        'csbsbs_symboliccompositecalc': understanding + object_use,
        'csbsbs_totalrawcalc': emotion + communication + gestures + sounds + words + understanding + object_use,
    }

print('Utility functions loaded.')

Utility functions loaded.


## CSBS BS (PID 6207) Validation

In [3]:
if TOKEN_6207:
    print('\n=== STEP 2: Validate CSBS BS Scoring (PID 6207) ===')

    metadata = post(TOKEN_6207, 'metadata')
    record_id_field = metadata[0]['field_name']
    score_fields = [
        'csbsbs_emotionraw','csbsbs_comraw','csbsbs_gesraw','csbsbs_soundsraw','csbsbs_wordsraw',
        'csbsbs_underraw','csbsbs_objectraw','csbsbs_socialcompositecalc','csbsbs_speechcompositecalc',
        'csbsbs_symboliccompositecalc','csbsbs_totalrawcalc'
    ]
    input_fields = [f'csbsbs_scale{i}' for i in range(1,16)] + [
        'csbsbs_scale16_1','csbsbs_scale16_2','csbsbs_scale16_3','csbsbs_scale17','csbsbs_scale18','csbsbs_scale19','csbsbs_scale20'
    ]
    fields = [record_id_field, 'redcap_event_name', 'csbs_bs_complete'] + input_fields + score_fields
    records = post(TOKEN_6207, 'record', type='flat', forms=['csbs_bs'], fields=fields, rawOrLabel='raw', rawOrLabelHeaders='raw')

    comparisons = 0
    mismatches = []
    for row in records:
        exp = bs_expected(row)
        if exp is None:
            continue
        rid = row.get(record_id_field, '')
        ev = row.get('redcap_event_name', '')
        for f, expected in exp.items():
            actual = num(row.get(f))
            if actual is None:
                mismatches.append((rid, ev, f, 'MISSING', expected))
                continue
            comparisons += 1
            if abs(actual - expected) > 1e-9:
                mismatches.append((rid, ev, f, actual, expected))

    print(f'Records: {len(records)}')
    print(f'Comparisons: {comparisons}')
    print(f'Mismatches: {len(mismatches)}')
    if mismatches:
        print('\n⚠ FIRST 10 MISMATCHES:')
        for rid, ev, f, actual, expected in mismatches[:10]:
            print(f'  {rid} / {ev} / {f}: actual={actual} vs expected={expected}')
    else:
        print('✓ PASS: All CSBS BS scores match approved logic')


=== STEP 2: Validate CSBS BS Scoring (PID 6207) ===


Records: 1767
Comparisons: 5533
Mismatches: 0
✓ PASS: All CSBS BS scores match approved logic


## Field Labels & Highlighting Check

In [4]:
print('\n=== STEP 3: Check Field Labels (AUTO-CALCULATED prefix) ===')

if TOKEN_6207:
    print('\n** CSBS BS (PID 6207) **')
    bs_score_fields = [
        'csbsbs_emotionraw','csbsbs_comraw','csbsbs_gesraw','csbsbs_soundsraw','csbsbs_wordsraw',
        'csbsbs_underraw','csbsbs_objectraw','csbsbs_socialcompositecalc','csbsbs_speechcompositecalc',
        'csbsbs_symboliccompositecalc','csbsbs_totalrawcalc'
    ]
    metadata = post(TOKEN_6207, 'metadata')
    by_field = {x['field_name']: x for x in metadata}
    
    bs_label_issues = []
    for f in bs_score_fields:
        meta = by_field.get(f)
        if not meta:
            bs_label_issues.append((f, 'NOT FOUND', ''))
            continue
        label = meta.get('field_label', '')
        calc = meta.get('select_choices_or_calculations', '')
        is_calc = meta.get('field_type') == 'calc'
        has_prefix = label.startswith('AUTO-CALCULATED:')
        if not has_prefix:
            bs_label_issues.append((f, label, 'MISSING AUTO-CALCULATED prefix'))
        else:
            print(f'  ✓ {f}: {label}')
    
    if bs_label_issues:
        print('\n⚠ CSBS BS Label Issues:')
        for f, label, issue in bs_label_issues:
            print(f'  {f}: {issue}')
    else:
        print('\n✓ All CSBS BS score fields have AUTO-CALCULATED prefix')


=== STEP 3: Check Field Labels (AUTO-CALCULATED prefix) ===

** CSBS BS (PID 6207) **


  ✓ csbsbs_emotionraw: AUTO-CALCULATED: Emotion and Eye Gaze Weighted Raw Score
  ✓ csbsbs_comraw: AUTO-CALCULATED: Communication Weighted Raw Score
  ✓ csbsbs_gesraw: AUTO-CALCULATED: Gestures Weighted Raw Score
  ✓ csbsbs_soundsraw: AUTO-CALCULATED: Sounds Weighted Raw Score
  ✓ csbsbs_wordsraw: AUTO-CALCULATED: Words Weighted Raw Score
  ✓ csbsbs_underraw: AUTO-CALCULATED: Understanding Weighted Raw Score
  ✓ csbsbs_objectraw: AUTO-CALCULATED: Object Use Weighted Raw Score
  ✓ csbsbs_socialcompositecalc: AUTO-CALCULATED: Social Composite Score
  ✓ csbsbs_speechcompositecalc: AUTO-CALCULATED: Speech Composite Score
  ✓ csbsbs_symboliccompositecalc: AUTO-CALCULATED: Symbolic Composite Score
  ✓ csbsbs_totalrawcalc: AUTO-CALCULATED: CSBS BS Total Raw Score

✓ All CSBS BS score fields have AUTO-CALCULATED prefix


## Manual Norm Fields Check (Visit-based)

In [5]:
print('\n=== STEP 4: Check Manual Norm Fields by Visit ===')

if TOKEN_6207:
    print('\n** CSBS BS (PID 6207) **')
    records = post(TOKEN_6207, 'record', type='flat', forms=['csbs_bs'], rawOrLabel='raw')
    
    # Group by event and check for manual norm fields
    events = {}
    for row in records:
        ev = row.get('redcap_event_name', 'N/A')
        if ev not in events:
            events[ev] = {'count': 0, 'complete': 0, 'with_norms': 0}
        events[ev]['count'] += 1
        if row.get('csbs_bs_complete') in ('1', '2'):  # complete or unverified
            events[ev]['complete'] += 1
    
    for ev, stats in sorted(events.items()):
        print(f'  Event: {ev} | Records: {stats["count"]} | Complete: {stats["complete"]}')


=== STEP 4: Check Manual Norm Fields by Visit ===

** CSBS BS (PID 6207) **


  Event: N/A | Records: 1767 | Complete: 597


## Consistency Summary

In [6]:
print('\n' + '='*60)
print('VALIDATION SUMMARY')
print('='*60)

if TOKEN_6207:
    if len(mismatches) == 0:
        print('✓ CSBS BS (PID 6207): PASS')
    else:
        print('⚠ CSBS BS (PID 6207): ISSUES FOUND')

if not TOKEN_6205:
    print('\n⚠ CSBS Caregiver (PID 6205): SKIPPED (token not set)')
    print('\nTo validate PID 6205, set: export REDCAP_TOKEN_6205=<your_token>')

print('\nFor more details, scroll up to see individual validation sections.')


VALIDATION SUMMARY
✓ CSBS BS (PID 6207): PASS

⚠ CSBS Caregiver (PID 6205): SKIPPED (token not set)

To validate PID 6205, set: export REDCAP_TOKEN_6205=<your_token>

For more details, scroll up to see individual validation sections.
